### Initialization

In [0]:

query = """
    SELECT 
        ROW_NUMBER() OVER(ORDER BY ci.customer_id) AS customer_key, -- surrogate key
        ci.customer_id,
        ci.customer_number,
        ci.first_name,
        ci.last_name,
        la.country,
        ci.marital_status,
        CASE 
            WHEN ci.gender <> 'n/a' THEN ci.gender -- crm is the master for gender
            ELSE COALESCE(ca.gender, 'n/a')
        END AS gender,
        ca.birth_date,
        ci.create_date
    FROM catalogue_project1.silver.crm_customers ci 
    LEFT JOIN catalogue_project1.silver.erp_cust_az12 ca
        ON      ci.customer_number = ca.customer_number
    LEFT JOIN catalogue_project1.silver.erp_loc_a101 la
        ON      ci.customer_number = la.customer_number
"""

### Check
That everything worked correctly.

In [0]:
df = spark.sql(query)
df.limit(10).display()

### Write into the Gold layer delta table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("catalogue_project1.gold.dim_customers")